---
## Stage 1 — Data Exploration

### What & Why

Before writing any model code, we need to understand the raw data deeply.  
BraTS2020 gives us 369 training cases, each with 4 MRI modalities and a segmentation mask.

**Key questions this stage answers:**
- What shape and dtype are the volumes?
- Are intensity ranges consistent across modalities and patients?
- What is the class distribution in the segmentation masks?
- What label remapping is required before training?

In [9]:
import nibabel as nib
import numpy as np
from pathlib import Path

# ── Point this at your BraTS2020 training data root ──────────────────────────
DATA_ROOT = Path(r"D:\personal projects\Brain-Tumor-Segmentation-with-BraTS\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData")
CASE_001  = DATA_ROOT / "BraTS20_Training_001"
MODALITIES = ["flair", "t1", "t1ce", "t2"]

### 1.1 Modality Exploration

We load each modality and inspect:
- **Shape** — should be `(240, 240, 155)` for all BraTS2020 volumes
- **Intensity range** — will differ across modalities (MRI values are not standardized)
- **Non-zero fraction** — tells us how much of the volume is actual brain vs background air

In [11]:
print("=" * 60)
print("MODALITY EXPLORATION — Case 001")
print("=" * 60)

for mod in MODALITIES:
    path = CASE_001 / f"BraTS20_Training_001_{mod}.nii"
    img  = nib.load(str(path))
    vol  = img.get_fdata()

    brain_mask    = vol > 0
    brain_voxels  = vol[brain_mask]

    print(f"\n── {mod.upper()} ──────────────────────────")
    print(f"  Shape:           {vol.shape}")
    print(f"  Dtype:           {vol.dtype}")
    print(f"  Global min/max:  {vol.min():.1f}  /  {vol.max():.1f}")
    print(f"  Brain mean:      {brain_voxels.mean():.1f}")
    print(f"  Brain std:       {brain_voxels.std():.1f}")
    print(f"  Non-zero voxels: {brain_mask.sum():,}  /  {vol.size:,}  ({100*brain_mask.mean():.1f}%)")

MODALITY EXPLORATION — Case 001

── FLAIR ──────────────────────────
  Shape:           (240, 240, 155)
  Dtype:           float64
  Global min/max:  0.0  /  625.0
  Brain mean:      173.0
  Brain std:       64.9
  Non-zero voxels: 1,342,885  /  8,928,000  (15.0%)

── T1 ──────────────────────────
  Shape:           (240, 240, 155)
  Dtype:           float64
  Global min/max:  0.0  /  678.0
  Brain mean:      354.3
  Brain std:       84.2
  Non-zero voxels: 1,342,885  /  8,928,000  (15.0%)

── T1CE ──────────────────────────
  Shape:           (240, 240, 155)
  Dtype:           float64
  Global min/max:  0.0  /  1845.0
  Brain mean:      417.3
  Brain std:       109.2
  Non-zero voxels: 1,342,885  /  8,928,000  (15.0%)

── T2 ──────────────────────────
  Shape:           (240, 240, 155)
  Dtype:           float64
  Global min/max:  0.0  /  376.0
  Brain mean:      114.7
  Brain std:       47.7
  Non-zero voxels: 1,342,885  /  8,928,000  (15.0%)


**What this tells us:**

| Observation | Implication |
|---|---|
| All modalities: shape `(240, 240, 155)` | Pre-registered — voxel [x,y,z] is the same tissue in all 4 |
| All modalities: same non-zero fraction | Shared brain mask — background zeroed identically |
| Intensity ranges differ wildly (T2 max=376 vs T1ce max=1845) | Cannot normalize globally — must normalize **per modality** |
| Only 15% of voxels are non-zero | 85% is background air — wasteful for model input, crop it |

### 1.2 Segmentation Mask Exploration

The segmentation mask is the ground truth we train against.  
BraTS2020 uses label set `{0, 1, 2, 4}` — note the jump from 2 to 4.  
This is a historical artifact. Our model uses output indices `{0,1,2,3}`, so we must remap `4 → 3`.

In [12]:
print("=" * 60)
print("SEGMENTATION MASK — Case 001")
print("=" * 60)

seg_path = CASE_001 / "BraTS20_Training_001_seg.nii"
seg = nib.load(str(seg_path)).get_fdata().astype(np.uint8)

CLASS_NAMES = {0: "Background", 1: "Necrotic Core (NCR)", 2: "Peritumoral Edema (ED)", 4: "Enhancing Tumor (ET)"}

print(f"\n  Shape:         {seg.shape}")
print(f"  Unique labels: {np.unique(seg)}")
print()

for label in np.unique(seg):
    count = int((seg == label).sum())
    pct   = 100 * count / seg.size
    name  = CLASS_NAMES.get(label, "Unknown")
    print(f"  Label {label}: {count:>10,} voxels  ({pct:.2f}%)  ← {name}")

print()
print("⚠  Label 4 will be remapped to 3 before training")
print(f"   Tumor burden: {100*(seg>0).mean():.2f}% of all voxels")

SEGMENTATION MASK — Case 001

  Shape:         (240, 240, 155)
  Unique labels: [0 1 2 4]

  Label 0:  8,716,021 voxels  (97.63%)  ← Background
  Label 1:     15,443 voxels  (0.17%)  ← Necrotic Core (NCR)
  Label 2:    168,794 voxels  (1.89%)  ← Peritumoral Edema (ED)
  Label 4:     27,742 voxels  (0.31%)  ← Enhancing Tumor (ET)

⚠  Label 4 will be remapped to 3 before training
   Tumor burden: 2.37% of all voxels


**Why class imbalance matters:**

Background = **97.63%** of all voxels. If a model predicts background everywhere,  
it achieves 97.63% voxel accuracy — and is clinically worthless.

This is why we use **Dice loss** instead of cross-entropy alone.  
Dice is computed per class independently, so a 0.17% class gets the same gradient weight as a 50% class.

**BraTS Evaluation Regions** (derived from the 4 labels):

| Region | Labels | Clinical meaning |
|---|---|---|
| Whole Tumor (WT) | {1, 2, 3} | Total tumor extent |
| Tumor Core (TC) | {1, 3} | Surgically targetable core |
| Enhancing Tumor (ET) | {3} | Active, contrast-enhancing tumor |

### 1.3 Affine and Voxel Size

In [13]:
img = nib.load(str(CASE_001 / "BraTS20_Training_001_t1.nii"))
print("Affine matrix (voxel → world space in mm):")
print(img.affine)
print()
voxel_size = np.sqrt((img.affine[:3, :3] ** 2).sum(axis=0))
print(f"Voxel size: {voxel_size} mm")
print("→ Isotropic 1mm³ — each voxel represents 1mm × 1mm × 1mm of brain tissue")

Affine matrix (voxel → world space in mm):
[[ -1.  -0.  -0.   0.]
 [ -0.  -1.  -0. 239.]
 [  0.   0.   1.   0.]
 [  0.   0.   0.   1.]]

Voxel size: [1. 1. 1.] mm
→ Isotropic 1mm³ — each voxel represents 1mm × 1mm × 1mm of brain tissue


---
## Stage 2 — Z-Score Normalization

### What & Why

MRI intensity values are **scanner-dependent** — they have no universal physical meaning.  
A voxel value of 400 in one patient's T1 is not comparable to 400 in another patient's T1,  
even on the same scanner.

**Why not min-max normalization?**  
MRI volumes often contain bright artifact voxels (motion, metal implants) that would  
compress the entire meaningful intensity range into a tiny interval.

**The solution: Z-score normalization restricted to brain voxels**

$$z = \frac{x - \mu_{brain}}{\sigma_{brain}}$$

Where $\mu_{brain}$ and $\sigma_{brain}$ are computed only over non-zero (brain) voxels.  
Background voxels are left at exactly 0.

**Applied independently per modality** — never across modalities, never globally.

### 2.1 Implementation

```
CODE                                    │  EXPLANATION
────────────────────────────────────────│─────────────────────────────────────────
def normalize_modality(vol):            │  Takes one 3D modality volume.
                                        │
    brain_mask = vol > 0                │  Boolean mask: True = brain tissue.
                                        │  Background air is exactly 0 in BraTS.
                                        │
    if brain_mask.sum() == 0:           │  Edge case: completely empty volume
        return vol                      │  (e.g. a corrupted scan). Return as-is,
                                        │  do not divide by zero.
                                        │
    mu  = vol[brain_mask].mean()        │  Mean of brain voxels ONLY.
    std = vol[brain_mask].std() + 1e-8  │  Std of brain voxels + epsilon.
                                        │  Epsilon (1e-8) prevents div-by-zero
                                        │  if a region has constant intensity.
                                        │
    normalized = np.zeros_like(vol)     │  Start with all zeros — background
                                        │  stays 0 without any extra masking step.
                                        │
    normalized[brain_mask] = (          │  Apply z-score to brain voxels only.
        vol[brain_mask] - mu) / std     │  Background is untouched (stays 0).
                                        │
    return normalized                   │  Returns float32 array, same shape.
```

In [14]:
def normalize_modality(vol: np.ndarray) -> np.ndarray:
    """
    Z-score normalization restricted to brain (non-zero) voxels.
    Background voxels remain exactly 0.
    Returns float32 array of same shape as input.
    """
    brain_mask = vol > 0

    if brain_mask.sum() == 0:
        return vol

    mu  = vol[brain_mask].mean()
    std = vol[brain_mask].std() + 1e-8

    normalized = np.zeros_like(vol)
    normalized[brain_mask] = (vol[brain_mask] - mu) / std
    return normalized.astype(np.float32)

### 2.2 Verification

In [15]:
vol  = nib.load(str(CASE_001 / "BraTS20_Training_001_t1ce.nii")).get_fdata().astype(np.float32)
norm = normalize_modality(vol)
brain_mask = vol > 0

print("=== BEFORE ===")
print(f"  mean (brain): {vol[brain_mask].mean():.2f}")
print(f"  std  (brain): {vol[brain_mask].std():.2f}")
print(f"  background:   {vol[0,0,0]:.4f}")

print("\n=== AFTER ===")
print(f"  mean (brain): {norm[brain_mask].mean():.4f}   ← should be ≈ 0.0")
print(f"  std  (brain): {norm[brain_mask].std():.4f}    ← should be ≈ 1.0")
print(f"  background:   {norm[0,0,0]:.4f}               ← should be exactly 0.0")
print(f"  dtype:        {norm.dtype}                    ← should be float32")

print("\n=== EDGE CASES ===")
empty  = np.zeros((240,240,155), dtype=np.float32)
result = normalize_modality(empty)
print(f"  Empty volume — all zeros: {(result == 0).all()}")
print(f"  Empty volume — no NaN:    {np.isfinite(result).all()}")
print(f"  Background unchanged:     {((vol==0) == (norm==0)).all()}")

print("\n=== ALL MODALITIES ===")
for mod in MODALITIES:
    v = nib.load(str(CASE_001 / f"BraTS20_Training_001_{mod}.nii")).get_fdata().astype(np.float32)
    n = normalize_modality(v)
    b = v > 0
    print(f"  {mod:6s} → mean: {n[b].mean():+.4f}  std: {n[b].std():.4f}")

=== BEFORE ===
  mean (brain): 417.33
  std  (brain): 109.19
  background:   0.0000

=== AFTER ===
  mean (brain): 0.0000   ← should be ≈ 0.0
  std  (brain): 1.0000    ← should be ≈ 1.0
  background:   0.0000               ← should be exactly 0.0
  dtype:        float32                    ← should be float32

=== EDGE CASES ===
  Empty volume — all zeros: True
  Empty volume — no NaN:    True
  Background unchanged:     True

=== ALL MODALITIES ===
  flair  → mean: +0.0000  std: 1.0000
  t1     → mean: -0.0000  std: 1.0000
  t1ce   → mean: +0.0000  std: 1.0000
  t2     → mean: +0.0000  std: 1.0000


---
## Stage 3 — Bounding Box Crop + Resize

### What & Why

After normalization the volume is still `(240, 240, 155)` — but 85% is background zeros.  
Feeding this to a 3D U-Net wastes GPU memory on empty space.

**Two-step spatial reduction:**
1. **Crop** — find the smallest box enclosing all non-zero voxels, discard the rest
2. **Resize** — interpolate the cropped volume to a fixed `(128, 128, 128)` shape

Why `128³`? It is the largest cube that fits in ~10GB VRAM with batch size 1  
using a standard 3D U-Net with 32 base filters. It is the de facto BraTS standard.

### 3.1 Crop to Brain Bounding Box

```
CODE                                    │  EXPLANATION
────────────────────────────────────────│─────────────────────────────────────────
def crop_to_brain(vol):                 │  Takes a 3D numpy array.
                                        │
    coords = np.array(                  │  np.where(vol > 0) returns a tuple of
        np.where(vol > 0))              │  3 arrays: (x_idxs, y_idxs, z_idxs)
                                        │  for every non-zero voxel.
                                        │  np.array(...) stacks them → shape (3, N)
                                        │
    if coords.shape[1] == 0:            │  Edge case: empty volume.
        return vol                      │
                                        │
    mins = coords.min(axis=1)           │  Minimum index along each axis.
                                        │  shape (3,) → [x_min, y_min, z_min]
                                        │
    maxs = coords.max(axis=1) + 1       │  Maximum index + 1.
                                        │  +1 because Python slicing is exclusive
                                        │  at the end: vol[0:228] gives indices
                                        │  0..227, so last brain voxel at 227
                                        │  requires stop index 228.
                                        │
    return vol[mins[0]:maxs[0],         │  Slice all three axes simultaneously.
               mins[1]:maxs[1],         │  Returns a VIEW — no data is copied
               mins[2]:maxs[2]]         │  until you modify it.
```

In [16]:
def crop_to_brain(vol: np.ndarray) -> np.ndarray:
    """
    Crop vol to the tight bounding box of non-zero voxels.
    If the volume is entirely zero, return it unchanged.
    """
    coords = np.array(np.where(vol > 0))

    if coords.shape[1] == 0:
        return vol

    mins = coords.min(axis=1)
    maxs = coords.max(axis=1) + 1

    return vol[mins[0]:maxs[0],
               mins[1]:maxs[1],
               mins[2]:maxs[2]]

### 3.2 Resize to Target Shape

```
CODE                                    │  EXPLANATION
────────────────────────────────────────│─────────────────────────────────────────
def resize_volume(vol, target=          │  Default target is 128³.
        (128,128,128)):                 │
                                        │
    tensor = torch.from_numpy(vol)      │  numpy → torch tensor
        .float()                        │  Ensure float32
        .unsqueeze(0)                   │  Add batch dim:   (H,W,D) → (1,H,W,D)
        .unsqueeze(0)                   │  Add channel dim: (1,H,W,D) → (1,1,H,W,D)
                                        │  F.interpolate requires (B,C,H,W,D)
                                        │
    resized = F.interpolate(            │
        tensor,                         │
        size=target,                    │  Target spatial shape (128,128,128)
        mode="trilinear",               │  3D equivalent of bilinear for images.
                                        │  Smoothly interpolates between voxels.
                                        │  (nearest-neighbor creates blocky edges)
        align_corners=True              │  Corner voxels of input map exactly to
    )                                   │  corner voxels of output.
                                        │  Use True for medical data — ensures
                                        │  image and mask stay aligned when resized
                                        │  separately.
                                        │
    return resized.squeeze().numpy()    │  Remove batch+channel dims, back to numpy
                                        │  (1,1,128,128,128) → (128,128,128)
```

In [17]:
import torch
import torch.nn.functional as F

def resize_volume(vol: np.ndarray, target=(128, 128, 128)) -> np.ndarray:
    """
    Resize a 3D volume to target shape using trilinear interpolation.
    align_corners=True ensures image and mask stay aligned when resized separately.
    """
    tensor = torch.from_numpy(vol).float().unsqueeze(0).unsqueeze(0)

    resized = F.interpolate(
        tensor,
        size=target,
        mode="trilinear",
        align_corners=True
    )

    return resized.squeeze().numpy()

### 3.3 Verification

In [18]:
vol  = nib.load(str(CASE_001 / "BraTS20_Training_001_t1ce.nii")).get_fdata().astype(np.float32)
norm = normalize_modality(vol)

print("=== CROP ===")
cropped = crop_to_brain(norm)
print(f"  Before: {norm.shape}")
print(f"  After:  {cropped.shape}   ← smaller than (240,240,155)")
print(f"  Brain signal preserved: {(cropped > 0).any()}")

print("\n=== RESIZE ===")
resized = resize_volume(cropped, target=(128, 128, 128))
print(f"  Shape: {resized.shape}   ← should be (128, 128, 128)")
print(f"  dtype: {resized.dtype}   ← should be float32")

print("\n=== FULL PIPELINE — ALL 4 MODALITIES ===")
print("  (normalize → crop → resize)")
for mod in MODALITIES:
    v = nib.load(str(CASE_001 / f"BraTS20_Training_001_{mod}.nii")).get_fdata().astype(np.float32)
    processed = resize_volume(crop_to_brain(normalize_modality(v)))
    print(f"  {mod:6s} → {processed.shape}  mean={processed[processed>0].mean():+.3f}")
print("\n  ✅ All modalities: (128, 128, 128) — ready for model input")

=== CROP ===
  Before: (240, 240, 155)
  After:  (136, 171, 132)   ← smaller than (240,240,155)
  Brain signal preserved: True

=== RESIZE ===
  Shape: (128, 128, 128)   ← should be (128, 128, 128)
  dtype: float32   ← should be float32

=== FULL PIPELINE — ALL 4 MODALITIES ===
  (normalize → crop → resize)
  flair  → (128, 128, 128)  mean=+0.932
  t1     → (128, 128, 128)  mean=+0.747
  t1ce   → (128, 128, 128)  mean=+0.659
  t2     → (128, 128, 128)  mean=+0.993

  ✅ All modalities: (128, 128, 128) — ready for model input


### 3.4 What the Pipeline Does to One Voxel

```
Raw T1ce voxel at brain center:   417.3  (scanner units, meaningless across patients)
After normalize_modality:           0.0  (mean of brain = 0, std = 1)
After crop_to_brain:                0.0  (same value, just in smaller array)
After resize_volume:               ~0.0  (trilinear blend of neighbors, close to 0)
```

```
Raw T1ce bright tumor voxel:     1845.0
After normalize_modality:          12.9  (13 standard deviations above brain mean)
After crop_to_brain:               12.9
After resize_volume:              ~12.0  (slightly smoothed by interpolation)
```

The tumor voxel remains a strong outlier even after normalization.  
That outlier signal is exactly what the model learns to detect.